> 对应 Lec 1-4。覆盖 Ray 全部核心 API 的可复用代码模板，以及蒙特卡洛/Bootstrap/交叉验证三类典型并行化例子，最后是跨组统计量合并与集群部署命令。概念含义见"2. Ray框架_概念解释.md"。

---

## Ray 初始化与关闭

Ray 启动时会在本机创建一个调度器、一个对象存储（Object Store）和若干工作进程，`ray.init()` 完成前这些组件都不可用。一个程序只能调用一次 `ray.init()`；`ray.shutdown()` 释放所有 Worker 进程和内存，通常在脚本结尾调用。

### ray.init() 本机模式与集群连接模式

In [ ]:
import ray

# ---- 本机模式 ----
ray.init()                  # 自动检测 CPU 数，最常用
ray.init(num_cpus=4)        # 显式指定使用 4 个 CPU

# ---- 连接已有集群 ----
# 在头节点上执行：ray start --head --num-cpus=1
# 在工作节点上执行：ray start --address=<头节点IP>:6379 --num-cpus=4
ray.init(address="auto")    # Python 端：自动发现集群，不创建新集群

### ray.shutdown() 调用时机与资源释放

In [ ]:
# 脚本末尾释放资源
ray.shutdown()

# 或用 try/finally 确保即使出错也能释放
try:
    ray.init()
    # ... 任务代码 ...
finally:
    ray.shutdown()

---

## @ray.remote 函数（Ray Task）模板

`@ray.remote` 把普通函数变成可以在远程进程上执行的任务。调用时用 `.remote()` 替代普通调用，**立即返回**一个 `ObjectRef`（类似快递单号），不等计算完成。

### 单返回值与多返回值写法（tuple return）

In [ ]:
import ray
import numpy as np

ray.init()

# ---- 单返回值 ----
@ray.remote
def compute_trace(X):
    return np.trace(X)

# ---- 多返回值（返回 tuple，用 ray.get 后再解包）----
@ray.remote
def compute_stats(X_chunk, y_chunk):
    XtX = X_chunk.T @ X_chunk
    Xty = X_chunk.T @ y_chunk
    n   = X_chunk.shape[0]
    return XtX, Xty, n           # 返回 tuple，不需要特殊声明

ref = compute_stats.remote(X_chunk, y_chunk)
XtX, Xty, n = ray.get(ref)      # ray.get 解包 tuple

### .remote() 调用 vs 直接调用的行为差异

In [ ]:
X = np.random.randn(1000, 500)

# 直接调用：立即执行，阻塞等待结果（串行）
result = compute_trace(X)        # 这里等待计算完成

# .remote() 调用：立即返回 ObjectRef，异步执行
ref = compute_trace.remote(X)    # 几乎瞬间返回，计算在后台进行
result = ray.get(ref)            # 这里才等待

---

## ray.get / ray.put 正确使用模式

`ray.get(ref)` 是**阻塞**操作：如果任务未完成，调用方就在此等待。并行性的关键在于"先批量分发，再统一等待"——所有任务在 `ray.get` 之前就已经开始并行运行了。

### 批量分发后统一 get（正确并行写法）

In [ ]:
@ray.remote
def slow_task(x):
    import time; time.sleep(1)
    return x * x

# ✅ 正确：先批量分发，全部任务同时运行，最后一次性等待
refs = [slow_task.remote(i) for i in range(8)]  # 8 个任务同时开始
results = ray.get(refs)                          # 等最慢的那个
# 总耗时 ≈ 1 秒（8 个任务并行运行）

### 循环内 get 导致串行退化（经典错误写法对比）

In [ ]:
# ❌ 错误：循环内立即 get，第 i+1 个任务等第 i 个完成后才开始
results = []
for i in range(8):
    ref = slow_task.remote(i)
    result = ray.get(ref)        # 阻塞等待，任务实际上是串行的
    results.append(result)
# 总耗时 ≈ 8 秒！

# ✅ 对比：正确写法总耗时 ≈ 1 秒
refs = [slow_task.remote(i) for i in range(8)]
results = ray.get(refs)

### ray.put 共享大数据（何时用、对比不用时的传输开销）

当同一份大数据被多个任务共用时，每次 `.remote()` 传参都会序列化并通过网络传输该数据。`ray.put()` 把数据一次性存入集群共享内存（Object Store），之后传递的只是一个引用 ID，零拷贝读取。

In [ ]:
# 场景：1000 个任务都要用同一个大矩阵 W（100MB）
W = np.random.randn(5000, 5000)

# ❌ 不用 put：W 被序列化并传输 1000 次，浪费 ~100GB 带宽
tasks = [compute_trace.remote(W) for _ in range(1000)]

# ✅ 用 put：W 存入共享内存一次，传递的是 ObjectRef（几十字节）
W_ref = ray.put(W)
tasks = [compute_trace.remote(W_ref) for _ in range(1000)]

# 何时用 ray.put：数据 > 几 MB 且被多个任务共用

---

## Map-Reduce 通用三阶段模板

Map-Reduce 是分布式计算的核心范式：Map 阶段并行处理各数据块，Barrier 统一等待，Reduce 阶段汇总结果。所有分布式算法（OLS、Logistic 回归、ADMM）都是这个模板的应用。

### Map 阶段：批量 .remote() 提交

In [ ]:
@ray.remote
def map_fn(data_chunk, params):
    """每个工作节点独立计算局部结果"""
    return local_compute(data_chunk, params)

# Map：列表推导式批量提交，所有任务立即并行开始
refs = [map_fn.remote(chunk, params) for chunk in data_chunks]

### Barrier：ray.get(refs) 统一等待

In [ ]:
# Barrier：等待所有 Map 任务完成，拿到全部结果
# all_results 是一个 Python 列表，每个元素是一个 Map 的返回值
all_results = ray.get(refs)

### Reduce 阶段：多返回值任务的分离汇总写法

In [ ]:
# 若 map_fn 返回 tuple (a, b, n)，需要分离后分别汇总
# 方法一：生成器表达式分离
total_a = sum(r[0] for r in all_results)
total_b = sum(r[1] for r in all_results)
total_n = sum(r[2] for r in all_results)

# 方法二：zip 解包（可读性好）
a_list, b_list, n_list = zip(*all_results)
total_a = sum(a_list)
total_n = sum(n_list)

# 矩阵汇总（元素级相加）
XtX = sum(r[0] for r in all_results)       # 等价于 np.sum([...], axis=0)

完整三阶段框架示例（分布式 X^TX）：

In [ ]:
import ray, numpy as np

ray.init()

@ray.remote
def compute_xtx(X_chunk_ref):
    X = X_chunk_ref              # ObjectRef 会自动解引用
    return X.T @ X

X = np.random.randn(100000, 500)
chunks = np.array_split(X, 10)

# Map
refs = [compute_xtx.remote(ray.put(chunk)) for chunk in chunks]
# Barrier
results = ray.get(refs)
# Reduce
XtX = sum(results)

ray.shutdown()

---

## Ray Actor 有状态节点模板

Ray Task 是**无状态**的：每次调用是独立的，函数执行完内存释放。Ray Actor 是**有状态**的：Actor 实例在整个生命周期中持久存在，状态保存在实例变量里，适合需要跨调用保持数据的场景（如 ADMM 工作节点持有本地数据 $X_i, y_i$）。

### Actor 类定义（@ray.remote class + __init__）

In [ ]:
@ray.remote
class WorkerNode:
    def __init__(self, X_chunk, y_chunk, rho):
        # 数据和状态在初始化时一次性传入，之后不再重传
        self.X = X_chunk
        self.y = y_chunk
        self.rho = rho
        self.XtX = X_chunk.T @ X_chunk    # 预计算，只做一次
        self.Xty = X_chunk.T @ y_chunk
        self.x_local = np.zeros(X_chunk.shape[1])
        self.u_local = np.zeros(X_chunk.shape[1])

    def get_local_stats(self, z):
        """接收全局 z，返回本地更新结果"""
        rhs = self.Xty + self.rho * (z - self.u_local)
        self.x_local = np.linalg.solve(self.XtX + self.rho * np.eye(len(z)), rhs)
        return self.x_local, self.u_local

    def update_dual(self, z):
        """更新对偶变量 u"""
        self.u_local += self.x_local - z
        return np.sum((self.x_local - z) ** 2)   # 局部残差平方

### Actor 创建（.remote()）与方法调用（method.remote()）

In [ ]:
# 创建 Actor 实例（异步，立即返回 ActorHandle）
worker = WorkerNode.remote(X_chunk, y_chunk, rho=1.0)

# 调用方法（也是异步的，返回 ObjectRef）
ref = worker.get_local_stats.remote(z)
x_i, u_i = ray.get(ref)         # 等待并取结果

# 批量创建多个 Actor
workers = [WorkerNode.remote(chunks[i], y_chunks[i], rho) for i in range(N)]

# 并行调用所有 Actor 的同一方法
refs = [w.get_local_stats.remote(z) for w in workers]
all_results = ray.get(refs)

### Task vs Actor 选择原则（何时需要持久化状态）

| 场景 | 选 Task | 选 Actor |
|------|---------|---------|
| 数据一次性处理（OLS Map） | ✅ | — |
| 迭代算法，数据在多轮迭代中复用 | — | ✅ |
| 需要跨调用积累状态（x_i, u_i 随迭代更新） | — | ✅ |
| 简单无状态的 Map-Reduce | ✅ | — |

---

## 蒙特卡洛模拟并行化（Monte Carlo π 估计）

**算法思路**：在 $[-1,1]\times[-1,1]$ 正方形内均匀撒点，落入单位圆内的比例约为 $\pi/4$，则 $\hat\pi = 4 \times \text{(圆内点数)} / \text{(总点数)}$。各组独立估计，最终取平均（等样本量）或加权平均（不等样本量）。

### 单组估计函数（固定种子 np.random.seed(group_id)）

In [ ]:
@ray.remote
def estimate_pi_chunk(group_id, n_points=1_000_000):
    np.random.seed(group_id)   # 每组用不同种子：保证各组独立且可重复
    x = np.random.uniform(-1.0, 1.0, n_points)
    y = np.random.uniform(-1.0, 1.0, n_points)
    in_circle = (x**2 + y**2 <= 1.0)
    return 4.0 * in_circle.sum() / n_points, n_points   # 返回 (估计值, 样本量)

### 各组样本量相同时直接取均值

In [ ]:
ray.init()

n_groups = 1000
tasks = [estimate_pi_chunk.remote(g) for g in range(1, n_groups + 1)]
results = ray.get(tasks)

estimates = [r[0] for r in results]
pi_hat = np.mean(estimates)
print(f"π ≈ {pi_hat:.6f}")

### 各组样本量不同时加权平均（np.average + weights）

**原理**：各组样本量 $n_i$ 不同时，直接取均值是错的（相当于给大组和小组相同权重）。正确做法是按样本量加权：$\hat\pi = \sum n_i \hat\pi_i / \sum n_i$。

In [ ]:
@ray.remote
def estimate_pi_variable(group_id):
    np.random.seed(group_id)
    n_points = group_id * 1000    # 各组样本量不同
    x = np.random.uniform(-1.0, 1.0, n_points)
    y = np.random.uniform(-1.0, 1.0, n_points)
    in_circle = (x**2 + y**2 <= 1.0)
    return 4.0 * in_circle.sum() / n_points, n_points

tasks = [estimate_pi_variable.remote(g) for g in range(1, 1001)]
results = ray.get(tasks)

estimates = np.array([r[0] for r in results])
weights   = np.array([r[1] for r in results])  # 样本量作为权重
pi_hat    = np.average(estimates, weights=weights)
# np.average(values, weights=w) 等价于 sum(w * values) / sum(w)

---

## Bootstrap 置信区间并行化

**算法思路**：有放回重抽样 B 次，每次计算统计量（如中位数），用 B 个统计量的经验分布来近似估计量的抽样分布，从中提取百分位数作为置信区间。B 次重抽样互相独立，天然适合 Map-Reduce。

### 分布式环境中随机种子的设置（为什么每个 Task 必须单独 seed）

不同 Ray Worker 进程是独立的 Python 进程，无法继承主进程的随机状态。若不显式 `np.random.seed()`，各进程可能用相同的初始随机状态，导致不同任务产生相同的 Bootstrap 样本，统计结论失效。

### 有放回重抽样 + ray.put 共享原始数据（避免重复传输）

In [ ]:
@ray.remote
def bootstrap_step(data_ref, seed):
    """
    data_ref: 通过 ray.put 存入共享内存的原始数据引用
    seed: 每次重抽样用不同种子，保证 B 次样本互相独立
    """
    np.random.seed(seed)
    data = data_ref                        # ObjectRef 自动解引用
    n = len(data)
    sample = np.random.choice(data, size=n, replace=True)  # 有放回重抽样
    return np.median(sample)              # 计算统计量（这里是中位数）

### 中位数 Bootstrap 完整代码（B 轮 Map-Reduce）

In [ ]:
ray.init()

original_data = np.random.randn(10000)    # 原始数据
data_ref = ray.put(original_data)         # 一次性存入共享内存

B = 1000
tasks = [bootstrap_step.remote(data_ref, seed=i) for i in range(B)]
bootstrap_medians = ray.get(tasks)        # 得到 B 个 Bootstrap 中位数

# 每次只传 data 的引用 ID（几十字节），不传 10000*8 字节的数据

### 置信区间提取（np.percentile 2.5% 和 97.5%）

In [ ]:
lower = np.percentile(bootstrap_medians, 2.5)
upper = np.percentile(bootstrap_medians, 97.5)
print(f"中位数的 95% Bootstrap CI: [{lower:.4f}, {upper:.4f}]")

---

## 网格搜索交叉验证并行化

**算法思路**：超参数组合数 × K 折 = 总任务数，每个（超参数组合，折）是独立的训练任务，可以全部并行提交。关键优化：大数据用 `ray.put` 存入共享内存，任务只传索引（小整数数组）而非数据切片。

### 展开所有嵌套循环（超参数 × K 折）批量提交

In [ ]:
from sklearn.model_selection import KFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

@ray.remote
def train_eval_fold(X_ref, y_ref, train_idx, val_idx, C, gamma):
    # X_ref/y_ref 已在共享内存，按索引切片是零拷贝读取
    X_train, X_val = X_ref[train_idx], X_ref[val_idx]
    y_train, y_val = y_ref[train_idx], y_ref[val_idx]
    model = SVC(C=C, gamma=gamma)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

ray.init()

X_ref = ray.put(X)   # 大数据只存一次
y_ref = ray.put(y)

C_values     = [0.1, 1.0, 10.0]
gamma_values = [0.001, 0.01, 0.1]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 展开三层嵌套循环，一次性提交所有任务
tasks      = []
param_keys = []   # 记录每个任务对应的超参数

for C in C_values:
    for gamma in gamma_values:
        for train_idx, val_idx in kf.split(X):
            tasks.append(train_eval_fold.remote(X_ref, y_ref, train_idx, val_idx, C, gamma))
        param_keys.append((C, gamma))

all_scores = ray.get(tasks)   # 等待所有任务

### 传索引而非传切片（配合 ray.put 的零拷贝写法）

`X_ref[train_idx]` 在 Worker 进程内执行，读取的是共享内存中的数据，不产生网络传输。若改为在 Driver 端先切片再传 `X_train`，则每个任务都要通过网络传输一份大矩阵，通信量 = 任务数 × 数据大小。

### 按每 K 个结果分组计算各超参数组合的均值

In [ ]:
K = 5
n_param_combos = len(param_keys)

best_score, best_params = -1, None
for i, (C, gamma) in enumerate(param_keys):
    fold_scores = all_scores[i * K : (i + 1) * K]   # 取第 i 个参数组合的 5 折分数
    mean_score  = np.mean(fold_scores)
    if mean_score > best_score:
        best_score, best_params = mean_score, (C, gamma)

print(f"最优参数: C={best_params[0]}, gamma={best_params[1]}, CV 分数: {best_score:.4f}")

---

## 跨组统计量合并

将数据分发到多个节点各自计算均值和方差，最终在主节点合并得到全局统计量。**均值可以加权平均，方差不能直接平均**（忽略组间方差会低估总方差）。

### 均值合并：加权平均（∑n_i·μ_i / ∑n_i，不能直接平均各组均值）

$$\hat\mu = \frac{\sum_{i=1}^m n_i \hat\mu_i}{\sum_{i=1}^m n_i}$$

直接取均值 `np.mean(mu_list)` 相当于给每组相同权重，当各组样本量 $n_i$ 不同时结果是错的。

### 方差合并：组内方差 + 组间方差（公式推导与代码）

$$\hat\sigma^2 = \frac{1}{N}\sum_{i=1}^m \left[n_i\hat\sigma_i^2 + n_i(\hat\mu_i - \hat\mu)^2\right]$$

两项含义：第一项 $n_i\hat\sigma_i^2$ 是组内方差贡献，第二项 $n_i(\hat\mu_i-\hat\mu)^2$ 是组间（各组均值与全局均值的偏差）贡献。缺少第二项会低估总体方差。

### 分布式任务同时返回 (n_i, μ_i, σ²_i) 三元组的写法

In [ ]:
@ray.remote
def compute_group_stats(group_id):
    n_i = 1000 * group_id
    np.random.seed(group_id)
    # 生成数据（示例：5 行 n 列矩阵，取每列 max-min）
    U = np.random.uniform(0, 1, size=(5, n_i))
    X = U.max(axis=0) - U.min(axis=0)
    return n_i, X.mean(), X.var()   # (样本量, 均值, 方差)

ray.init()

m = 1000
tasks   = [compute_group_stats.remote(i) for i in range(1, m + 1)]
results = ray.get(tasks)

ns   = np.array([r[0] for r in results])   # 各组样本量
mus  = np.array([r[1] for r in results])   # 各组均值
vars_= np.array([r[2] for r in results])   # 各组方差

N      = ns.sum()
mu_hat = np.sum(ns * mus) / N                                   # 加权均值
var_hat= np.sum(ns * vars_ + ns * (mus - mu_hat) ** 2) / N     # 组内 + 组间

print(f"μ̂ = {mu_hat:.6f}, σ̂² = {var_hat:.6f}")

ray.shutdown()

---

## Ray 集群部署命令

Ray 集群由一个头节点（Head Node）和若干工作节点（Worker Node）组成。头节点运行调度器和 Driver 程序；工作节点提供计算资源。

### head 节点启动（ray start --head --num-cpus）

```bash
# 在头节点机器上执行
ray start --head --num-cpus=1
# 输出中会显示：To add another node to this Ray cluster, run
#   ray start --address='192.168.1.1:6379' --num-cpus=...
```

`--num-cpus=1` 表示头节点只用 1 个 CPU 做调度（不参与计算），保留调度资源。

### worker 节点加入（ray start --address=<IP>:<port>）

```bash
# 在每个工作节点上执行，地址来自头节点启动时的输出
ray start --address='192.168.1.1:6379' --num-cpus=4

# 验证集群状态
ray status
```

### Python 侧连接已有集群（ray.init() 自动发现）

In [ ]:
import ray

# 在集群的头节点上运行 Python 脚本时，ray.init() 自动连接本地集群
ray.init()                        # 自动发现，不创建新集群

# 显式指定地址（在非头节点上运行时使用）
ray.init(address='192.168.1.1:6379')

# 检查集群资源
print(ray.cluster_resources())    # 输出所有节点的 CPU/GPU 总量
print(ray.available_resources())  # 输出当前空闲资源